# Single Image Analysis - Refactored API

This notebook demonstrates the complete workflow using the refactored `colokroll` package structure.

**Steps:**
1. Load image and metadata
2. Background subtraction per channel (CUDA-accelerated)
3. Cell segmentation via Cellpose API
4. Colocalization analysis with mask filtering
5. Export results as DataFrames


In [ ]:
from pathlib import Path
import time
import logging

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v3 as iio

# Refactored imports
from colokroll import (
    ImageLoader,
    ImageIOConfig,
    BackgroundSubtractor,
)
from colokroll.analysis.segmentation import get_hf_token
from colokroll.analysis.colocalization import (
    compute_colocalization,
    export_colocalization_json,
    estimate_min_area_threshold,
)
from gradio_client import Client, handle_file

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')


## 1. Load Image and Rename Channels

In [ ]:
# Update this path to your image file
image_path = Path("/path/to/your/image.ome.tiff")

image_loader = ImageLoader(ImageIOConfig(verbose=True))
loaded_data = image_loader.load_image(image_path)
pixel_size = image_loader.get_pixel_size()

print(f"Loaded image shape: {loaded_data.shape}")
print(f"Pixel size: {pixel_size} μm")
print(f"Original channel names: {image_loader.get_channel_names()}")

# Rename channels to match your experiment
new_channel_names = ['DAPI', 'ALIX', 'Phalloidin', 'LAMP1']
image_loader.rename_channels(new_channel_names)
channel_names = image_loader.get_channel_names()
print(f"Renamed channels: {channel_names}")


## 2. Background Subtraction (CUDA-accelerated)

The `BackgroundSubtractor` automatically searches for the best method and parameters.


In [ ]:
bg_subtractor = BackgroundSubtractor()

results = {}

for i, ch in enumerate(channel_names):
    ch_data = loaded_data[:, :, :, i]
    t0 = time.perf_counter()
    
    corrected, meta = bg_subtractor.subtract_background(
        image=ch_data,
        channel_name=ch,
        # method omitted -> auto search + full run
    )
    
    cp.cuda.Stream.null.synchronize()
    dt = time.perf_counter() - t0
    
    results[ch] = (corrected, meta)
    print(f"\n{ch}: {dt:.2f}s -> {meta['method']}")

### Visualize Background Subtraction Results


In [ ]:
middle_slice_idx = loaded_data.shape[0] // 2

fig = bg_subtractor.plot_background_subtraction_comparison(
    original_data=loaded_data,
    corrected_results=results,
    channel_names=channel_names,
    z_slice=middle_slice_idx,
    figsize=(5 * len(channel_names), 12)
)
plt.show()


## 3. Cell Segmentation via Cellpose API

Build a composite from Phalloidin + DAPI channels and segment using the Cellpose Gradio Space.


In [ ]:
# Build composite for segmentation
def norm01(a):
    a = a.astype(np.float32)
    mn, mx = a.min(), a.max()
    return np.zeros_like(a) if mx <= mn else (a - mn) / (mx - mn)

ph_idx = channel_names.index("Phalloidin")
da_idx = channel_names.index("DAPI")

ph_mip = loaded_data[..., ph_idx].max(axis=0).astype(np.float32)
da_mip = loaded_data[..., da_idx].max(axis=0).astype(np.float32)

composite = 0.8 * norm01(ph_mip) + 0.2 * norm01(da_mip)
composite = np.clip(np.nan_to_num(composite, nan=0.0, posinf=1.0, neginf=0.0), 0, 1).astype(np.float32)

# Save temporary PNG for Cellpose
tmp_png = "/tmp/composite.png"
iio.imwrite(tmp_png, (composite * 255).astype(np.uint8))

# Authenticated client
token = get_hf_token()
client = Client("mouseland/cellpose", hf_token=token)

# Two-step flow with pause + retry
def run_seg(resize):
    _ = client.predict(filepath=handle_file(tmp_png), api_name="/update_button")
    time.sleep(1.0)
    return client.predict(
        filepath=[handle_file(tmp_png)],
        resize=resize,
        max_iter=250,
        flow_threshold=0.4,
        cellprob_threshold=0.0,
        api_name="/cellpose_segment",
    )

result = None
for rs in (600, 400):
    try:
        result = run_seg(rs)
        break
    except Exception as e:
        print(f"Retry with smaller resize due to: {e}")
        time.sleep(1.0)

if result is None:
    raise RuntimeError("Cellpose Space failed after retries")

# Extract outputs
masks_tif = result[2]["value"] if isinstance(result[2], dict) else (
    result[2].path if hasattr(result[2], "path") else result[2]
)
outlines_png = result[3]["value"] if isinstance(result[3], dict) else (
    result[3].path if hasattr(result[3], "path") else result[3]
)

mask = iio.imread(str(masks_tif)).astype(np.int32)

# Save masks
save_dir = Path("./outputs/cellpose")
save_dir.mkdir(parents=True, exist_ok=True)
dst_mask = save_dir / f"{image_path.stem}_phall_dapi_masks.tif"
dst_outl = save_dir / f"{image_path.stem}_phall_dapi_outlines.png"
Path(dst_mask).write_bytes(Path(masks_tif).read_bytes())
Path(dst_outl).write_bytes(Path(outlines_png).read_bytes())

print(f"Mask saved to: {dst_mask}")

# Visualize mask
plt.figure(figsize=(6, 6))
plt.title("Mask (labels)")
plt.imshow(mask, cmap="tab20")
plt.axis("off")
plt.show()


## 4. Colocalization Analysis

Compute colocalization between two channels (e.g., ALIX and LAMP1) using the segmentation mask.


In [ ]:
# Load the mask we just created
mask_path = dst_mask

# Estimate minimum area threshold
min_area = estimate_min_area_threshold(mask_path, fraction_of_median=0.70)

# Run colocalization
res = compute_colocalization(
    image=results,  # dict[str, (array, meta)] or dict[str, array]
    mask=mask_path,
    channel_a="ALIX",
    channel_b="LAMP1",
    thresholding='costes',
    max_border_fraction=0.10,
    min_area=int(min_area),
    border_margin_px=1,
    plot_mask=True,
)

print("\nTotal image metrics:")
print(res["results"]["total_image"])


## 5. Export Results as DataFrames

In [ ]:
# Per-cell dataframe (one row per kept label)
df_cells = pd.DataFrame(res["results"]["per_label"]).sort_values("label")

# Total-image (single row)
df_total = pd.DataFrame([res["results"]["total_image"]])

# Optional: save to CSV
# df_cells.to_csv("per_cell_metrics.csv", index=False)
# df_total.to_csv("total_image_metrics.csv", index=False)

print("Per-cell metrics:")
df_cells


In [ ]:
print("Total image metrics:")
df_total

## Summary

This notebook demonstrated:
- Loading images with the new `colokroll.io.ImageLoader`
- Background subtraction with `colokroll.preprocessing.BackgroundSubtractor`
- Cell segmentation via Cellpose API
- Colocalization analysis with mask filtering
- Exporting results as DataFrames

All imports now use the refactored package structure:
- `colokroll.io` for image loading
- `colokroll.preprocessing` for background subtraction
- `colokroll.analysis.segmentation` for segmentation utilities
- `colokroll.analysis.colocalization` for colocalization metrics
